In [ ]:
import os, re
import numpy as np
import scanpy as sc
from os.path import join
import pandas as pd

import sys
import scipy.io as sio
import paste as pst
import scipy.sparse as sps
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

sys.path.insert(0, './')
from metrics import compute_PAS2, compute_CHAOS2

In [ ]:
def collect_ads(_dir):
    fds = os.listdir(_dir)
    ads = []
    for fd in fds:
        if fd.startswith("R"):
            continue
        fns = os.listdir(join(_dir, fd))
        adx = sc.read_h5ad(join(_dir, fd, fns[0]))
        adxh = sc.AnnData(np.zeros((adx.n_obs, 2)), obs=adx.obs, obsm={"spatial":adx.obsm['spatial']})
        ads.append(adxh)
    return ads

In [ ]:
def calc_PAS_CHAOS(data_dir, cluster_dir, ds, batch_key):
    ads = collect_ads(join(data_dir, ds))
    
    df_res = []
    for meth in target_methods:
        cluster_file = join(cluster_dir, meth, ds, 'df_cluster.csv')
        if not os.path.exists(cluster_file):
            print(f"[Skip] {cluster_file} not found.")
            continue

        df_cluster = pd.read_csv(cluster_file, index_col=0)
        
        for adx in ads:
            common_idx = adx.obs_names.intersection(df_cluster.index)
            adx_sub = adx[common_idx].copy()
            adx_sub.obs['cluster'] = df_cluster.loc[common_idx, 'cluster'].to_list()
            
            sample_name = adx_sub.obs[batch_key][0]
            pas = compute_PAS2(adx_sub, 'cluster', spatial_key='spatial')
            chaos = compute_CHAOS2(adx_sub, 'cluster', spatial_key='spatial')

            df_res.append({
                'sample': sample_name,
                'method': meth,
                'PAS': pas,
                'CHAOS': chaos
            })

    df_res = pd.DataFrame(df_res)
    return df_res

In [ ]:
target_methods = [
    'CLUE',
    'StabMap',
    'SpaMosaic(non-spatial)',
    'UINMF',
    'scMoMaT',
    'SpaMosaic',
    'MIDAS',
    'Cobolt',
    'MultiVI'
]

data_dir = '../../data/processed/'
cluster_dir = '../../results/clusters/'
score_dir = '../../results/scores/'

### DBiT-RnaAtac-Atac

In [ ]:
ds = 'DBiT-RnaAtac-Atac'

dfl = calc_PAS_CHAOS(data_dir, cluster_dir, ds, 'Slice')
dfl.to_csv(join(score_dir, ds, 'df_chaos_pas.csv'))

### Embryo

In [ ]:
ds = 'Embryo'

dfl = calc_PAS_CHAOS(data_dir, cluster_dir, ds, 'Slice')
dfl.to_csv(join(score_dir, ds, 'df_chaos_pas.csv'))

### Lymph-S1-S2-S3

In [ ]:
ds = 'Lymph-S1-S2-S3'

dfl = calc_PAS_CHAOS(data_dir, cluster_dir, ds, 'src')
dfl.to_csv(join(score_dir, ds, 'df_chaos_pas.csv'))

### MB-5M

In [ ]:
ds = 'MB-5M'

dfl = calc_PAS_CHAOS(data_dir, cluster_dir, ds, 'src')
dfl.to_csv(join(score_dir, ds, 'df_chaos_pas.csv'))

### MB-H3K4me3

In [ ]:
ds = 'MB-H3K4me3'

dfl = calc_PAS_CHAOS(data_dir, cluster_dir, ds, 'src')
dfl.to_csv(join(score_dir, ds, 'df_chaos_pas.csv'))

### Misar-E13-E15-E18

In [ ]:
ds = 'Misar-E13-E15-E18'

dfl = calc_PAS_CHAOS(data_dir, cluster_dir, ds, 'Sample')
dfl.to_csv(join(score_dir, ds, 'df_chaos_pas.csv'))

### Misar-Stereo

In [ ]:
ds = 'Misar-Stereo'

dfl = calc_PAS_CHAOS(data_dir, cluster_dir, ds, 'Sample')
dfl.to_csv(join(score_dir, ds, 'df_chaos_pas.csv'))

### Mux-VisiumHD-HE

In [ ]:
ds = 'Mux-VisiumHD-HE'

dfl = calc_PAS_CHAOS(data_dir, cluster_dir, ds, 'Slice')
dfl.to_csv(join(score_dir, ds, 'df_chaos_pas.csv'))

### Tonsil-S1-S2-S3

In [ ]:
ds = 'Tonsil-S1-S2-S3'

dfl = calc_PAS_CHAOS(data_dir, cluster_dir, ds, 'src')
dfl.to_csv(join(score_dir, ds, 'df_chaos_pas.csv'))

### Simulations

In [ ]:
def collect_obs2(data_dir):
    ad1 = sc.read_h5ad(join(data_dir, 'ad_rna1.h5ad'))
    ad2 = sc.read_h5ad(join(data_dir, 'ad_rna2.h5ad'))
    ad3 = sc.read_h5ad(join(data_dir, 'ad_adt2.h5ad'))
    ad1.obs['src'] = 's1'
    ad2.obs['src'] = 's2'
    ad3.obs['src'] = 's3'
    
    ads = [ad1, ad2, ad3]
    new_ads = []
    for adx in ads:
        adxh = sc.AnnData(np.zeros((adx.n_obs, 2)), obs=adx.obs, obsm={"spatial":adx.obsm['spatial']})
        new_ads.append(adxh)
    return new_ads

In [ ]:
ds = 'Simulations'
fds = os.listdir(join(data_dir, ds))
batch_key = 'src'

for fd in fds:
    for seed in ['1', '2', '3']:
        ads = collect_obs2(join(data_dir, ds, fd, seed))

        df_res = []
        for meth in target_methods:
            cluster_file = join(cluster_dir, meth, ds, fd, seed, 'df_cluster.csv')
            if not os.path.exists(cluster_file):
                print(f"[Skip] {cluster_file} not found.")
                continue

            df_cluster = pd.read_csv(cluster_file, index_col=0)

            for adx in ads:
                common_idx = adx.obs_names.intersection(df_cluster.index)
                adx_sub = adx[common_idx].copy()
                adx_sub.obs['cluster'] = df_cluster.loc[common_idx, 'cluster'].to_list()

                sample_name = adx_sub.obs[batch_key][0]
                pas = compute_PAS2(adx_sub, 'cluster', spatial_key='spatial')
                chaos = compute_CHAOS2(adx_sub, 'cluster', spatial_key='spatial')

                df_res.append({
                    'sample': sample_name,
                    'method': meth,
                    'PAS': pas,
                    'CHAOS': chaos
                })

        df_res = pd.DataFrame(df_res)
        df_res.to_csv(join(score_dir, ds, fd, seed, 'df_chaos_pas.csv'))